In [4]:
# ==================== FULL PIPELINE ====================
import os, random, numpy as np, pandas as pd, cv2
from glob import glob
from tqdm import tqdm
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import GroupShuffleSplit
from PIL import Image, ImageDraw, ImageFont
from collections import Counter
from torch.amp import GradScaler, autocast
import math

# Reproducibility
SEED = 123
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True
torch.backends.cuda.enable_flash_sdp = True 

# =================== Configuration ===================
DATA_DIR = '/kaggle/input/competitions/dl-lab-4-ocr'
TRAIN_IMG_DIR = os.path.join(DATA_DIR, 'train/train')
TEST_IMG_DIR = os.path.join(DATA_DIR, 'test/test')
TRAIN_CSV = os.path.join(DATA_DIR, 'train.csv')
OUTPUT_CSV = 'submission.csv'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_H, IMG_W = 32, 96
BATCH_SIZE = 128
LR, LR_MIN = 8e-4, 1e-6
WEIGHT_DECAY = 1e-4
SMOOTHING = 0.01
EPOCHS = 49
BLANK_IDX = 10
ALPHABET = '0123456789'
CHAR2IDX = {c: i for i, c in enumerate(ALPHABET)}

# =================== Data Loading ===================
train_df = pd.read_csv(TRAIN_CSV)
train_df['Price'] = train_df['Price'].astype(str)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, val_idx = next(gss.split(train_df, groups=train_df['Price']))
X_train = train_df['Filename'].iloc[train_idx].tolist()
y_train = train_df['Price'].iloc[train_idx].tolist()
X_val = train_df['Filename'].iloc[val_idx].tolist()
y_val = train_df['Price'].iloc[val_idx].tolist()
print(f'Train: {len(X_train)}, Val: {len(X_val)}')
print(f'Unique val prices: {len(set(y_val))}, Overlap: {len(set(y_val) & set(y_train))}')

# =================== Utilities ===================
def encode_price(price_str):
    return [CHAR2IDX[c] for c in price_str]

def decode_ctc_greedy(indices):
    result, prev = [], BLANK_IDX
    for idx in indices:
        if idx != BLANK_IDX and idx != prev:
            result.append(ALPHABET[idx])
        prev = idx
    return ''.join(result)

# =================== Synthetic Data Generation ===================
def load_fonts():
    font_paths = [
        '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
        '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf',
        '/usr/share/fonts/truetype/liberation/LiberationSans.ttf',
        '/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf',
        '/usr/share/fonts/truetype/freefont/FreeMono.ttf',
        '/usr/share/fonts/truetype/freefont/FreeSans.ttf',
    ]
    fonts = []
    for path in font_paths:
        try:
            fonts.append(ImageFont.truetype(path, size=24))
        except:
            pass
    return fonts or [ImageFont.load_default()]


def apply_realistic_defects(img, prob=0.3):
    if random.random() > prob:
        return img
    h, w = img.shape
    
    # 1. Non-uniform background
    if random.random() < 0.3:
        noise = cv2.GaussianBlur(np.random.randint(-10, 10, (h, w)).astype(np.float32), (21, 21), 0)
        img = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    
    # 2. Erosion/dilation
    if random.random() < 0.3:
        kernel = np.ones((random.choice([2, 3]),) * 2, np.uint8)
        img = cv2.erode(img, kernel, iterations=1) if random.random() < 0.5 else cv2.dilate(img, kernel, iterations=1)
    
    # 3. Motion blur
    if random.random() < 0.2:
        length = random.randint(3, 7)
        angle = random.choice([0, 90])
        kernel_mb = np.zeros((length, length))
        if angle == 0:
            kernel_mb[length // 2, :] = 1.0 / length
        else:
            kernel_mb[:, length // 2] = 1.0 / length
        img = cv2.filter2D(img, -1, kernel_mb)
    
    # 4. Blobs
    if random.random() < 0.2:
        for _ in range(random.randint(1, 3)):
            cv2.circle(img, (random.randint(0, w - 1), random.randint(0, h - 1)), random.randint(1, 4), random.choice([0, 255]), -1)
    
    # 5. Contrast reduction
    if random.random() < 0.3:
        mean = np.mean(img)
        img = np.clip((img - mean) * random.uniform(0.7, 0.95) + mean, 0, 255).astype(np.uint8)
    
    return img

def render_price(price, fonts, font_sizes):
    bg, fg = random.randint(220, 255), random.randint(0, 60)
    w, h = random.randint(50, 110), random.randint(26, 38)
    img = Image.new('L', (w, h), bg)
    draw = ImageDraw.Draw(img)
    font = ImageFont.truetype(random.choice(fonts).path, size=random.choice(font_sizes))
    bbox = draw.textbbox((0, 0), price, font=font)
    x, y = max(0, (w - (bbox[2] - bbox[0])) // 2), max(0, (h - (bbox[3] - bbox[1])) // 2)
    draw.text((x, y), price, fill=fg, font=font)
    img_np = np.array(img, dtype=np.uint8)
    if random.random() < 0.2:
        img_np = cv2.GaussianBlur(img_np, (3, 3), 0)
    if random.random() < 0.15:
        img_np = np.clip(img_np + np.random.randint(0, 15, img_np.shape, dtype=np.uint8), 0, 255)
    return apply_realistic_defects(img_np, prob=0.35)

def generate_general_synthetic(num_samples=12000):
    fonts, font_sizes = load_fonts(), [22, 24, 26, 28, 30]
    images, labels = [], []
    for _ in range(num_samples):
        r = random.random()
        length = 1 if r < 0.007 else (2 if r < 0.460 else (3 if r < 0.965 else 4))
        price = ''.join(random.choices('0123456789', k=length))
        images.append(render_price(price, fonts, font_sizes))
        labels.append(price)
    return images, labels

def generate_targeted_synthetic():
    fonts, font_sizes = load_fonts(), [22, 24, 26, 28, 30]
    images, labels = [], []
    
    # 2-digit with '8' at start
    for _ in range(2000):
        price = '8' + random.choice('0123456789')
        images.append(render_price(price, fonts, font_sizes))
        labels.append(price)
    
    # 3-digit with '2' in second position
    for _ in range(2000):
        price = random.choice('123456789') + '2' + random.choice('0123456789')
        images.append(render_price(price, fonts, font_sizes))
        labels.append(price)
    
    # 3-digit with '1' in first position
    for _ in range(1500):
        price = '1' + random.choice('23456789') + random.choice('0123456789')
        images.append(render_price(price, fonts, font_sizes))
        labels.append(price)
    
    # Single digits
    for _ in range(1000):
        price = str(random.randint(1, 9))
        images.append(render_price(price, fonts, font_sizes))
        labels.append(price)
    
    # More 2-digit confusion samples
    for _ in range(1500):
        price = random.choice('123456789') + random.choice('0123456789')
        images.append(render_price(price, fonts, font_sizes))
        labels.append(price)
    
    # Confusion with 0 and 9 endings
    for _ in range(5000):
        length = random.choice([2, 3])
        price = random.choice('0123456789') + random.choice('09')
        if length == 3:
            price = random.choice('0123456789') + price
        images.append(render_price(price, fonts, font_sizes))
        labels.append(price)
    
    return images, labels

print("Generating synthetic data...")
synthetic_images, synthetic_labels = generate_general_synthetic(12000)
targeted_images, targeted_labels = generate_targeted_synthetic()
print(f"Generated: {len(synthetic_images)} general, {len(targeted_images)} targeted")

# =================== Augmentations ===================
train_transforms = A.Compose([
    # ── Extreme geometric ──
    A.Rotate(limit=20, p=0.5, border_mode=cv2.BORDER_CONSTANT, fill=255),
    A.Affine(scale=(0.8, 1.1), p=0.3, border_mode=cv2.BORDER_CONSTANT, fill=255),
    A.MotionBlur(blur_limit=(5, 9), p=0.3),
    A.Perspective(scale=(0.03, 0.08), p=0.5),

    # ── Heavy blur / low resolution ──
    A.GaussianBlur(blur_limit=(4, 9), p=0.3),
    A.Downscale(scale_range=(0.7, 0.9), p=0.5),
    
    # ── Strong contrast / brightness changes ──
    A.CLAHE(clip_limit=3.0, tile_grid_size=(8,8), p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),

    # ── Partial occlusions (simulate cropped digits) ──
    A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(3, 12),
                    hole_width_range=(3, 12), fill=255, p=0.4),

    # ── Mild elastic / grid distortions (keep digits readable) ──
    A.ElasticTransform(alpha=2, sigma=30, p=0.2),
    A.GridDistortion(num_steps=5, distort_limit=0.1, p=0.2),

    # ── Noise ──
    A.GaussNoise(std_range=(0.01, 0.05), p=0.4),
    # ── Finalise ──
    A.Resize(height=IMG_H, width=IMG_W),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2(),
])

base_transforms = A.Compose([
    A.Resize(height=IMG_H, width=IMG_W),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2(),
])

# =================== Datasets ===================
class PriceDataset(Dataset):
    def __init__(self, filenames, labels, img_dir, transform=None):
        self.labels, self.transform, self.filenames = labels, transform, filenames
        self.cache = {}
        for fname in tqdm(filenames, desc="Caching images"):
            img = cv2.imread(os.path.join(img_dir, fname), cv2.IMREAD_GRAYSCALE)
            self.cache[fname] = img if img is not None else np.ones((20, 60), dtype=np.uint8) * 255

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        img = np.expand_dims(self.cache[self.filenames[idx]], axis=-1)
        if self.transform:
            img = self.transform(image=img)['image']
        return img, torch.tensor(encode_price(self.labels[idx]), dtype=torch.long)

class SyntheticDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images, self.labels, self.transform = images, labels, transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = np.expand_dims(self.images[idx], axis=-1)
        if self.transform:
            img = self.transform(image=img)['image']
        return img, torch.tensor(encode_price(self.labels[idx]), dtype=torch.long)

# Balance real data
X_train_aug, y_train_aug = [], []
for fname, label in zip(X_train, y_train):
    reps = 5 if len(label) == 1 else (3 if len(label) == 4 else 1)
    X_train_aug.extend([fname] * reps)
    y_train_aug.extend([label] * reps)

train_real = PriceDataset(X_train_aug, y_train_aug, TRAIN_IMG_DIR, train_transforms)
train_synth = SyntheticDataset(synthetic_images, synthetic_labels, train_transforms)
targeted_dataset = SyntheticDataset(targeted_images, targeted_labels, train_transforms)
train_dataset = ConcatDataset([train_real, train_synth, targeted_dataset])

val_dataset = PriceDataset(X_val, y_val, TRAIN_IMG_DIR, base_transforms)

print(f"Train size: real={len(train_real)}, synth={len(train_synth)}, total={len(train_dataset)}")

# Confusion pairs and auxiliary labels
CONFUSION_PAIRS = [(2, 3), (6, 9), (8, 9), (1, 3), (1, 7), (4, 6), (5, 6)]

def get_aux_labels(price_indices, pairs):
    present = set(price_indices)
    labels = torch.full((len(pairs),), -100, dtype=torch.long)
    for i, (a, b) in enumerate(pairs):
        has_a, has_b = a in present, b in present
        if has_a and not has_b:
            labels[i] = 0
        elif has_b and not has_a:
            labels[i] = 1
    return labels

def collate_fn_train(batch):
    images, labels = zip(*batch)
    images = torch.stack(images, dim=0)
    target_lengths = torch.tensor([len(l) for l in labels], dtype=torch.long)
    labels_cat = torch.cat(labels)
    aux_labels = torch.stack([get_aux_labels(l.tolist(), CONFUSION_PAIRS) for l in labels])
    return images, labels_cat, target_lengths, aux_labels

def collate_fn_val(batch):
    images, labels = zip(*batch)
    images = torch.stack(images, dim=0)
    target_lengths = torch.tensor([len(l) for l in labels], dtype=torch.long)
    return images, torch.cat(labels), target_lengths

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn_train, num_workers=4, persistent_workers=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        collate_fn=collate_fn_val, num_workers=2, persistent_workers=True, pin_memory=True)

# =================== Model Architecture ===================
class ECABlock(nn.Module):
    def __init__(self, channels, gamma=2, b=1):
        super().__init__()
        # Adaptive kernel size
        t = int(abs((math.log2(channels) + b) / gamma))
        k = t if t % 2 else t + 1
        self.conv = nn.Conv1d(1, 1, kernel_size=k, padding=k//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: (B, C, H, W)
        y = x.mean(dim=[2, 3], keepdim=True)        # (B, C, 1, 1)
        y = y.squeeze(-1).transpose(-1, -2)          # (B, 1, C)
        y = self.conv(y)                             # (B, 1, C)
        y = y.transpose(-1, -2).unsqueeze(-1)        # (B, C, 1, 1)
        return x * self.sigmoid(y)

class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, use_eca=False):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.downsample = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, stride=stride),
            nn.BatchNorm2d(out_ch)
        ) if stride != 1 or in_ch != out_ch else None
        self.se = ECABlock(out_ch) if use_eca else None

    def forward(self, x):
        identity = x
        out = nn.functional.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        out = identity + out
        if self.se:
            out = self.se(out)
        return nn.functional.relu(out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7, temperature=1.0, attn_dropout=0.2):
        super().__init__()
        self.temperature = temperature
        self.conv = nn.Conv2d(2, 1, kernel_size=(1, kernel_size), padding=(0, kernel_size // 2), bias=False)
        self.dropout = nn.Dropout2d(attn_dropout) if attn_dropout > 0 else nn.Identity()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        attn = torch.sigmoid(self.dropout(self.conv(torch.cat([avg_out, max_out], dim=1)) / self.temperature))
        return x * attn

class PositionalEncoding1D(nn.Module):
    def __init__(self, max_time_steps, d_model):
        super().__init__()
        self.max_T = max_time_steps
        self.pos_embed = nn.Parameter(torch.randn(max_time_steps, d_model) * 0.02)

    def forward(self, x):
        T = x.size(0)
        return x + self.pos_embed[:T].unsqueeze(1)

class DigitDiscriminationHead(nn.Module):
    def __init__(self, in_channels, pair_list, dropout=0.3):
        super().__init__()
        self.pairs = pair_list
        self.num_pairs = len(pair_list)
        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_channels, in_channels // 2),    # 512 → 256
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(in_channels // 2, in_channels // 4), # 256 → 128
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(in_channels // 4, self.num_pairs * 2)  # 128 → num_pairs*2
        )
    def forward(self, x):
        return self.fc(x).view(-1, self.num_pairs, 2)

class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, nhead, num_layers=2, dropout=0.2):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=dropout, activation='gelu', batch_first=False
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers, enable_nested_tensor=False)

    def forward(self, x):
        return self.encoder(x)


class MixStyle(nn.Module):
    """MixStyle regularisation for feature maps."""
    def __init__(self, p=0.5, alpha=0.1, eps=1e-6):
        super().__init__()
        self.p = p
        self.alpha = alpha          # beta distribution parameter (like mixup)
        self.eps = eps

    def forward(self, x):
        # x: (B, C, H, W)
        if not self.training or torch.rand(1).item() > self.p:
            return x
        B, C, H, W = x.shape
        if B < 2:
            return x

        # Randomly shuffle the batch to pair each sample with another
        index = torch.randperm(B, device=x.device)
        
        # Instance-level mean and std
        mu = x.mean(dim=[2, 3], keepdim=True)          # (B, C, 1, 1)
        var = x.var(dim=[2, 3], keepdim=True, unbiased=False)
        sig = (var + self.eps).sqrt()

        # Get mean and std of the paired sample
        mu_shuffled = mu[index]
        sig_shuffled = sig[index]

        # Sample mixing coefficients lambda from Beta(alpha, alpha)
        lambda_mu = torch.distributions.Beta(self.alpha, self.alpha).sample((B, C, 1, 1)).to(x.device)
        lambda_sig = torch.distributions.Beta(self.alpha, self.alpha).sample((B, C, 1, 1)).to(x.device)

        # Mixed statistics
        mu_mix = lambda_mu * mu + (1 - lambda_mu) * mu_shuffled
        sig_mix = lambda_sig * sig + (1 - lambda_sig) * sig_shuffled

        # Normalize x with its own statistics and re-normalize with mixed statistics
        x_norm = (x - mu) / sig
        x_mixed = x_norm * sig_mix + mu_mix
        return x_mixed

class CRNN_Enhanced(nn.Module):
    def __init__(self, img_h=32, img_w=96, num_classes=11, use_spatial_attn=True,
                 use_pos_embed=True, max_time_steps=48, use_transformer=False,
                 transformer_layers=2, nhead=8, use_aux_head=True, pair_list=None,
                 use_mixstyle=True, mixstyle_p=0.5, mixstyle_alpha=0.1,
                 seq_dropout=0.1, aux_loss_weight=0.05, cnn_dropout=0.1): 
        super().__init__()
        self.use_transformer = use_transformer
        self.use_aux_head = use_aux_head
        self.aux_loss_weight = aux_loss_weight
        self.mixstyle = MixStyle(p=mixstyle_p, alpha=mixstyle_alpha) if use_mixstyle else nn.Identity()
        
        self.cnn_stem = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            ResidualBlock(64, 64, stride=2),
            ResidualBlock(64, 128, stride=2),
        )
        self.cnn_late = nn.Sequential(
            ResidualBlock(128, 256, stride=(2, 2)),
            ResidualBlock(256, 512, stride=(2, 1), use_eca=True),
            ResidualBlock(512, 512, stride=(2, 1), use_eca=True),
        )
        
        # 1×1 conv to project early features to 512 channels
        self.fusion_conv = nn.Conv2d(128, 512, kernel_size=1)

        # ============ NEW: Dropout2d layers ============
        # Dropout after early CNN stem (before fusion)
        self.dropout_early = nn.Dropout2d(cnn_dropout) if cnn_dropout > 0 else nn.Identity()
        # Dropout after fusion and before spatial attention
        self.dropout_fusion = nn.Dropout2d(cnn_dropout) if cnn_dropout > 0 else nn.Identity()
        # Dropout after spatial attention (before sequence model)
        self.dropout_late = nn.Dropout2d(cnn_dropout * 0.5) if cnn_dropout > 0 else nn.Identity()
        # ================================================

        self.spatial_attn = SpatialAttention(kernel_size=7) if use_spatial_attn else None
        self.seq_dropout = nn.Dropout2d(seq_dropout) if seq_dropout > 0 else nn.Identity()
        self.pos_encoder = PositionalEncoding1D(max_time_steps, 512) if use_pos_embed else None

        if use_transformer:
            self.seq_model = TransformerEncoderBlock(d_model=512, nhead=nhead, num_layers=transformer_layers)
        else:
            self.rnn = nn.GRU(512, 256, num_layers=2, bidirectional=True, batch_first=False, dropout=0.2)

        self.fc = nn.Linear(512, num_classes)

        if use_aux_head:
            self.aux_head = DigitDiscriminationHead(512, pair_list)

    def forward(self, x, return_aux_logits=False):
        # Early CNN (before width compression)
        feats_early = self.cnn_stem(x)              # (B, 128, H_early, T_early)

        # ✅ Dropout2d #1: After early stem
        feats_early = self.dropout_early(feats_early)
        feats_early = self.mixstyle(feats_early)
        # Late CNN
        feats = self.cnn_late(feats_early)          # (B, 512, 1, T)
        
        # Multi-scale fusion: project early features and add to late features
        feats_early_proj = self.fusion_conv(feats_early)  # (B, 512, H_early, T_early)
        feats_early_proj = nn.functional.adaptive_avg_pool2d(feats_early_proj, feats.shape[2:])
        feats = feats + feats_early_proj              # fused features
        
        # ✅ Dropout2d #2: After fusion
        feats = self.dropout_fusion(feats)
        
        # Spatial attention
        if self.spatial_attn is not None:
            feats = self.spatial_attn(feats)
        
        # ✅ Dropout2d #3: After spatial attention (lighter dropout)
        feats = self.dropout_late(feats)
        
        # Existing feature map dropout (seq_dropout)
        feats = self.seq_dropout(feats)
        
        # Prepare sequence
        feats_seq = feats.squeeze(2).permute(2, 0, 1)
        if self.pos_encoder is not None:
            feats_seq = self.pos_encoder(feats_seq)
        
        # Sequence model
        if self.use_transformer:
            seq_out = self.seq_model(feats_seq)
        else:
            self.rnn.flatten_parameters()
            seq_out, _ = self.rnn(feats_seq)
        
        # Classifier
        logits = self.fc(seq_out)
        
        # Auxiliary head
        if self.use_aux_head and return_aux_logits:
            aux_logits = self.aux_head(feats_seq.mean(dim=0))
            return logits, aux_logits
        return logits

model = CRNN_Enhanced(
    IMG_H, IMG_W, num_classes=11, 
    use_transformer=True, transformer_layers=2,
    nhead=8, use_aux_head=True, pair_list=CONFUSION_PAIRS, 
    seq_dropout=0.1, aux_loss_weight=0.05,
    cnn_dropout=0.1, use_mixstyle=True, mixstyle_p=0.5, mixstyle_alpha=0.2
).to(DEVICE)

# =================== Loss and Optimizer ===================
def ctc_loss_fn(logits, targets, input_lengths, target_lengths):
    log_probs = logits.log_softmax(dim=2)
    loss = nn.functional.ctc_loss(
        log_probs, targets, input_lengths, target_lengths,
        blank=BLANK_IDX, reduction='mean', zero_infinity=True
    )
    if SMOOTHING > 0:
        loss = (1 - SMOOTHING) * loss + SMOOTHING * (-log_probs.mean(dim=2).mean())
    return loss

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# =================== Training ===================
def compute_loss(model, imgs, labels, target_lengths, aux_labels=None):
    logits, aux_logits = model(imgs, return_aux_logits=True)
    input_lengths = torch.full((imgs.size(0),), logits.size(0), dtype=torch.long, device=DEVICE)
    loss_main = ctc_loss_fn(logits, labels, input_lengths, target_lengths)    
    
    if model.use_aux_head and aux_labels is not None:
        valid_mask = (aux_labels != -100).float()
        ce_per_pair = nn.functional.cross_entropy(
            aux_logits.permute(0, 2, 1), aux_labels.clamp(min=0), reduction='none'
        )
        aux_loss = (ce_per_pair * valid_mask).sum() / (valid_mask.sum() + 1e-8)
        return loss_main + model.aux_loss_weight * aux_loss, loss_main, aux_loss
    return loss_main, loss_main, torch.tensor(0.0, device=DEVICE)

@torch.inference_mode()
def evaluate(model, loader, device):
    model.eval()
    correct = total = 0
    
    pbar = tqdm(loader, desc='Val', leave=False)
    for imgs, labels, target_lengths in pbar:
        imgs = imgs.to(device)
        
        with autocast('cuda'):
            logits = model(imgs)
        
        _, max_idx = logits.max(dim=2)
        max_idx = max_idx.transpose(0, 1).cpu().numpy()
        labels_np = labels.cpu().numpy()
        tlen = target_lengths.cpu().numpy()
        
        start = 0
        for i, L in enumerate(tlen):
            true_idx = labels_np[start:start + L]
            start += L
            pred_str = decode_ctc_greedy(max_idx[i])
            true_str = ''.join(ALPHABET[idx] for idx in true_idx)
            if pred_str == true_str:
                correct += 1
            total += 1
        
        current_acc = correct / total if total > 0 else 0
        pbar.set_postfix({'acc': f'{current_acc:.4f}'})
    
    return correct / total

best_acc = 0.0
scaler = GradScaler('cuda')
steps_per_epoch = len(train_loader)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=7 * steps_per_epoch, T_mult=2, eta_min=LR_MIN
)

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = running_main = running_aux = 0.0
    
    pbar = tqdm(train_loader, desc=f'Train Epoch {epoch}', leave=False)
    for batch_idx, (imgs, labels, target_lengths, aux_labels) in enumerate(pbar):
        global_step = (epoch - 1) * steps_per_epoch + batch_idx
        
        # LR warmup
        if global_step < 3 * steps_per_epoch:
            new_lr = LR * (global_step / (3 * steps_per_epoch))
        else:
            scheduler.step(global_step)
            new_lr = scheduler.get_last_lr()[0]
        for pg in optimizer.param_groups:
            pg['lr'] = new_lr
        
        imgs, labels, target_lengths, aux_labels = (
            imgs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True),
            target_lengths.to(DEVICE, non_blocking=True), aux_labels.to(DEVICE, non_blocking=True)
        )
        optimizer.zero_grad(set_to_none=True)
        
        with autocast('cuda'):
            loss, loss_main, aux_loss = compute_loss(model, imgs, labels, target_lengths, aux_labels)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()
        
        bs = imgs.size(0)
        running_loss += loss.item() * bs
        running_main += loss_main.item() * bs
        running_aux += aux_loss.item() * bs
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'main': f'{loss_main.item():.4f}', 'aux': f'{aux_loss.item():.4f}'})
    
    n = len(train_loader.dataset)
    val_acc = evaluate(model, val_loader, DEVICE)
    print(f'Epoch {epoch:02d} | LR {optimizer.param_groups[0]["lr"]:.2e} | '
          f'Loss {running_loss/n:.4f} | Main {running_main/n:.4f} | Aux {running_aux/n:.4f} | '
          f'Val Acc {val_acc:.4f}')
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({
            'epoch': epoch, 'model_state_dict': model.state_dict(), 'best_acc': best_acc,
        }, 'best_model_mixstyle.pth')
        print(f'  ✅ New best (acc: {best_acc:.4f})')

print(f'\n🏆 Best Val Accuracy: {best_acc:.4f}')

Train: 12454, Val: 2596
Unique val prices: 33, Overlap: 0
Generating synthetic data...
Generated: 12000 general, 13000 targeted


Caching images: 100%|██████████| 2596/2596 [00:02<00:00, 1174.32it/s]


Train size: real=12840, synth=12000, total=37840


Epoch 01 | LR 2.66e-04 | Loss 2.0626 | Main 2.0316 | Aux 0.6193 | Val Acc 0.8286
  ✅ New best (acc: 0.8286)


Epoch 02 | LR 5.32e-04 | Loss 0.7283 | Main 0.7121 | Aux 0.3248 | Val Acc 0.8505
  ✅ New best (acc: 0.8505)


...

Epoch 45 | LR 4.06e-05 | Loss 0.2564 | Main 0.2535 | Aux 0.0580 | Val Acc 0.9807


Epoch 46 | LR 2.35e-05 | Loss 0.2617 | Main 0.2588 | Aux 0.0594 | Val Acc 0.9819


Epoch 47 | LR 1.11e-05 | Loss 0.2588 | Main 0.2558 | Aux 0.0585 | Val Acc 0.9819


Epoch 48 | LR 3.53e-06 | Loss 0.2546 | Main 0.2517 | Aux 0.0585 | Val Acc 0.9823
  ✅ New best (acc: 0.9823)


Epoch 49 | LR 1.00e-06 | Loss 0.2521 | Main 0.2493 | Aux 0.0578 | Val Acc 0.9819

🏆 Best Val Accuracy: 0.9823


In [5]:
# =================== Beam Search (pyctcdecode) ===================
DATA_DIR = '/kaggle/input/competitions/dl-lab-4-ocr'
TRAIN_IMG_DIR = os.path.join(DATA_DIR, 'train/train')
TEST_IMG_DIR = os.path.join(DATA_DIR, 'test/test')
TRAIN_CSV = os.path.join(DATA_DIR, 'train.csv')
SUB_CSV = os.path.join(DATA_DIR, 'sample_submission.csv')
OUTPUT_CSV = 'submission.csv'

try:
    from pyctcdecode import build_ctcdecoder
    decoder = build_ctcdecoder(labels=list(ALPHABET), kenlm_model_path=None)
    def decode_ctc_beam(logits_tensor, beam_width=5):
        logits_np = logits_tensor.squeeze(1).cpu().numpy()
        result = decoder.decode(logits_np, beam_width=beam_width)
        if not result or len(result) < 1 or len(result) > 4:
            _, max_idx = logits_tensor.squeeze(1).max(dim=1)
            result = decode_ctc_greedy(max_idx.cpu().numpy())
        return result
except ImportError:
    print("pyctcdecode не установлен, использую ручной beam search")
    def decode_ctc_beam(logits_tensor, beam_width=5, min_len=1, max_len=4):
        logits = logits_tensor.squeeze(1).cpu().numpy()
        T, C = logits.shape
        log_probs = logits - np.max(logits, axis=1, keepdims=True)
        log_probs = log_probs - np.log(np.sum(np.exp(log_probs), axis=1, keepdims=True) + 1e-10)
        hypotheses = {('', BLANK_IDX): 0.0}
        for t in range(T):
            next_hypotheses = {}
            frame_probs = log_probs[t]
            for (prefix_tuple, last_char), hyp_prob in hypotheses.items():
                prefix = list(prefix_tuple)
                for c in range(C):
                    if frame_probs[c] < -10: continue
                    prob = frame_probs[c]
                    if c == BLANK_IDX:
                        key = (tuple(prefix), BLANK_IDX)
                        next_hypotheses[key] = np.logaddexp(next_hypotheses.get(key, -np.inf), hyp_prob + prob)
                    elif c == last_char and last_char != BLANK_IDX:
                        key = (tuple(prefix), c)
                        next_hypotheses[key] = np.logaddexp(next_hypotheses.get(key, -np.inf), hyp_prob + prob)
                    elif c != BLANK_IDX:
                        new_prefix = tuple(prefix + [c])
                        key = (new_prefix, c)
                        next_hypotheses[key] = np.logaddexp(next_hypotheses.get(key, -np.inf), hyp_prob + prob)
            if next_hypotheses:
                sorted_hyps = sorted(next_hypotheses.items(), key=lambda x: x[1], reverse=True)
                hypotheses = dict(sorted_hyps[:beam_width])
        valid_hyps = {}
        for (prefix_tuple, last_char), prob in hypotheses.items():
            prefix_len = len([c for c in prefix_tuple if c != BLANK_IDX])
            if min_len <= prefix_len <= max_len:
                valid_hyps[(prefix_tuple, last_char)] = prob
        if not valid_hyps: valid_hyps = hypotheses
        best_key, _ = max(valid_hyps.items(), key=lambda x: x[1])
        best_chars = [c for c in best_key[0] if c != BLANK_IDX]
        return ''.join([ALPHABET[i] for i in best_chars])

# =================== Сабмит ===================
checkpoint = torch.load('/kaggle/working/best_model_mixstyle.pth', map_location=DEVICE)
if 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✅ Loaded fine-tuned model (best acc: {checkpoint.get('best_acc', 0):.4f})")
else:
    model.load_state_dict(checkpoint)
    print("✅ Loaded model weights")

model.eval()

submission = pd.read_csv(SUB_CSV)
preds = []
for fname in tqdm(submission['Filename'], desc='Predicting'):
    img = cv2.imread(os.path.join(TEST_IMG_DIR, fname), cv2.IMREAD_GRAYSCALE)
    img = np.expand_dims(img, axis=-1)
    transformed = base_transforms(image=img)['image'].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(transformed)
        pred_str = decode_ctc_beam(logits, beam_width=5)
    preds.append(pred_str)

submission['Price'] = preds
submission.to_csv(OUTPUT_CSV, index=False)
print('Submission saved to', OUTPUT_CSV)

pyctcdecode не установлен, использую ручной beam search
✅ Loaded fine-tuned model (best acc: 0.9823)


Predicting: 100%|██████████| 3762/3762 [00:52<00:00, 70.98it/s]

Submission saved to submission.csv
